# Progree Data Analytics Internship — Task 4
## Statistical Inferential Hypothesis Analysis & Predictive Business Forecaster

**Dataset:** Online Retail II  
**Source:** UCI Machine Learning Repository, DOI 10.24432/C5CG6D  
**Environment:** Python / Pandas / SciPy / Statsmodels / Matplotlib / Seaborn

This project follows the Task 4 requirements: formal significance testing using an A/B-style two-group validation, one-way ANOVA, and Chi-Square analysis; followed by SARIMA time-series forecasting with holdout evaluation and 95% confidence intervals. The A/B-style comparison is observational rather than a randomized experiment, and this limitation is documented explicitly.

## 1. Dataset Source
The Online Retail II dataset contains 1,067,371 transaction records from a UK-based non-store online retailer covering 01/12/2009 to 09/12/2011. UCI classifies it as multivariate, sequential, and time-series data and identifies regression among its associated tasks. citeturn0search0

**Citation:** Chen, D. (2012). *Online Retail II*. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.

In [ ]:
# Install once if needed
# %pip install pandas numpy scipy statsmodels matplotlib seaborn openpyxl ucimlrepo

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.multicomp import pairwise_tukeyhsd

BASE = Path.cwd()
DATA_FILE = BASE / "online_retail_II.xlsx"

if DATA_FILE.exists():
    df = pd.concat(pd.read_excel(DATA_FILE, sheet_name=None).values(), ignore_index=True)
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=502)
    df = dataset.data.features.copy()

# Normalize common source column variants
rename = {"Invoice":"InvoiceNo", "Price":"UnitPrice", "Customer ID":"CustomerID"}
df = df.rename(columns=rename)
df.columns = ['InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
print('Raw shape:', df.shape)

## 2. Analytical Cleaning
Exact duplicate rows are removed. Cancellation invoices, non-positive quantities, non-positive prices, invalid dates, and missing customer IDs are excluded from the customer-level positive-sales analytical view. Revenue is calculated as `Quantity × UnitPrice`.

In [ ]:
df = df.drop_duplicates().copy()
df['IsCancellation'] = df['InvoiceNo'].astype(str).str.upper().str.startswith('C')
df['Revenue'] = df['Quantity'] * df['UnitPrice']
sales = df[(~df['IsCancellation']) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['InvoiceDate'].notna()].copy()
sales['CustomerID'] = sales['CustomerID'].astype('string')
sales = sales[sales['CustomerID'].notna()].copy()
sales['Month'] = sales['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
print('Analytical rows:', len(sales))
print('Customers:', sales['CustomerID'].nunique())
print('Orders:', sales['InvoiceNo'].nunique())
print('Revenue:', sales['Revenue'].sum())

## 3. Customer RFM Segmentation for Inferential Testing

In [ ]:
snapshot = sales['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = sales.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()
rfm['RScore'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['FScore'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['MScore'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
score = rfm[['RScore','FScore','MScore']].sum(axis=1)
rfm['Segment'] = pd.cut(score,[0,5,8,10,12,15],labels=['Hibernating','Needs Attention','Potential Loyalists','Loyal Customers','Champions']).astype(str)
rfm.loc[(rfm.RScore>=4)&(rfm.FScore<=2),'Segment']='Recent Customers'
rfm.loc[(rfm.RScore<=2)&(rfm.FScore>=3),'Segment']='At Risk'
rfm['Segment'].value_counts()

## 4. A/B-Style Two-Group Significance Validation
**H0:** Mean customer monetary value is equal for UK and non-UK customers.

**H1:** The means differ.

A Welch two-sample t-test is used because the groups have unequal sample sizes and no equal-variance assumption is imposed. A 95% confidence interval is reported for the mean difference. This is an observational two-group comparison, not a randomized A/B experiment.

In [ ]:
cust_country = sales.groupby('CustomerID').agg(
    Country=('Country', lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
    Monetary=('Revenue','sum')
).reset_index()
gA = cust_country.loc[cust_country.Country.eq('United Kingdom'),'Monetary']
gB = cust_country.loc[~cust_country.Country.eq('United Kingdom'),'Monetary']
t_stat, p_value = stats.ttest_ind(gA, gB, equal_var=False, nan_policy='omit')
mean_diff = gA.mean() - gB.mean()
se = np.sqrt(gA.var(ddof=1)/len(gA) + gB.var(ddof=1)/len(gB))
df_w = (gA.var(ddof=1)/len(gA) + gB.var(ddof=1)/len(gB))**2 / ((gA.var(ddof=1)/len(gA))**2/(len(gA)-1) + (gB.var(ddof=1)/len(gB))**2/(len(gB)-1))
ci = stats.t.interval(0.95, df_w, loc=mean_diff, scale=se)
ab_result = pd.DataFrame([{'Test':'Welch two-sample t-test','Group A':'United Kingdom','Group B':'Non-UK','N_A':len(gA),'N_B':len(gB),'Mean_A':gA.mean(),'Mean_B':gB.mean(),'Mean_Difference':mean_diff,'t':t_stat,'df':df_w,'p_value':p_value,'CI95_Lower':ci[0],'CI95_Upper':ci[1],'Decision':'Reject H0' if p_value<0.05 else 'Fail to reject H0'}])
ab_result

## 5. One-Way ANOVA
**H0:** All RFM segments have the same mean monetary value.

**H1:** At least one segment mean differs.

The ANOVA is followed by Tukey HSD pairwise comparisons when significant. Eta-squared is reported as an effect-size measure.

In [ ]:
groups = [g['Monetary'].values for _, g in rfm.groupby('Segment', observed=True)]
F_stat, p_anova = stats.f_oneway(*groups)
grand = rfm['Monetary'].mean()
ss_between = sum(len(g)*(g['Monetary'].mean()-grand)**2 for _,g in rfm.groupby('Segment', observed=True))
ss_total = ((rfm['Monetary']-grand)**2).sum()
eta_squared = ss_between/ss_total
anova_result = pd.DataFrame([{'Test':'One-way ANOVA','F_statistic':F_stat,'p_value':p_anova,'Eta_squared':eta_squared,'Decision':'Reject H0' if p_anova<0.05 else 'Fail to reject H0'}])
anova_result

tukey = pairwise_tukeyhsd(rfm['Monetary'], rfm['Segment'], alpha=0.05)
tukey_df = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
tukey_df.head()

## 6. Chi-Square Test of Independence
**H0:** Country group and RFM segment are independent.

**H1:** Country group and RFM segment are associated.

To avoid sparse categories, the five countries with the most customers are retained and all remaining countries are grouped as `Other`.

In [ ]:
top5 = cust_country['Country'].value_counts().head(5).index
cc = cust_country.copy()
cc['CountryGroup'] = np.where(cc['Country'].isin(top5), cc['Country'], 'Other')
merged_cc = cc.set_index('CustomerID').join(rfm.set_index('CustomerID')['Segment'], how='inner')
contingency = pd.crosstab(merged_cc['CountryGroup'], merged_cc['Segment'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
chi_result = pd.DataFrame([{'Test':'Chi-Square independence','Chi2_statistic':chi2,'df':dof,'p_value':p_chi,'Min_Expected_Count':expected.min(),'Decision':'Reject H0' if p_chi<0.05 else 'Fail to reject H0'}])
chi_result

## 7. Monthly Business Forecasting with SARIMA
The target is monthly revenue. The final five months are held out as a test set. A SARIMA model with monthly seasonality is fitted on the training period. Forecast intervals are reported at 95%. Performance is evaluated using MAE, RMSE and MAPE.

In [ ]:
monthly = sales.groupby('Month').agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'), Quantity=('Quantity','sum')).sort_index().asfreq('MS').fillna(0)
h = 5
train, test = monthly['Revenue'].iloc[:-h], monthly['Revenue'].iloc[-h:]
model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)
pred = fit.get_forecast(steps=h)
pred_mean = pred.predicted_mean
pred_ci = pred.conf_int(alpha=0.05)
mae = np.mean(np.abs(test-pred_mean))
rmse = np.sqrt(np.mean((test-pred_mean)**2))
mape = np.mean(np.abs((test-pred_mean)/test))*100
forecast_test = pd.DataFrame({'Month':test.index,'Actual_Revenue':test.values,'Forecast_Revenue':pred_mean.values,'Lower_95':pred_ci.iloc[:,0].values,'Upper_95':pred_ci.iloc[:,1].values})
performance = pd.DataFrame([{'Model':'SARIMA(1,1,1)x(1,1,1,12)','MAE':mae,'RMSE':rmse,'MAPE_percent':mape}])
performance

In [ ]:
# Refit on all available history and forecast the next six months
full_fit = SARIMAX(monthly['Revenue'], order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
future = full_fit.get_forecast(steps=6)
future_mean = future.predicted_mean
future_ci = future.conf_int(alpha=0.05)
future_forecast = pd.DataFrame({'Month':future_mean.index,'Forecast_Revenue':future_mean.values,'Lower_95':future_ci.iloc[:,0].values,'Upper_95':future_ci.iloc[:,1].values})
future_forecast

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(train.index, train.values, label='Train')
plt.plot(test.index, test.values, label='Actual Test')
plt.plot(pred_mean.index, pred_mean.values, label='Forecast')
plt.fill_between(pred_mean.index, pred_ci.iloc[:,0], pred_ci.iloc[:,1], alpha=.2, label='95% CI')
plt.title('SARIMA Revenue Forecast — Holdout Evaluation'); plt.xlabel('Month'); plt.ylabel('Revenue'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(11,5))
plt.plot(monthly.index, monthly.Revenue, label='Historical Revenue')
plt.plot(future_mean.index, future_mean.values, label='Future Forecast')
plt.fill_between(future_mean.index, future_ci.iloc[:,0], future_ci.iloc[:,1], alpha=.2, label='95% CI')
plt.title('Future Monthly Revenue Forecast with 95% Confidence Interval'); plt.xlabel('Month'); plt.ylabel('Revenue'); plt.legend(); plt.tight_layout(); plt.show()

## 8. Export Results
Export the statistical test tables, contingency table, monthly business metrics, holdout forecast, model performance, and future forecast to CSV files for submission.

In [ ]:
OUT = BASE/'outputs'; OUT.mkdir(exist_ok=True)
ab_result.to_csv(OUT/'ab_test_results.csv', index=False)
anova_result.to_csv(OUT/'anova_results.csv', index=False)
tukey_df.to_csv(OUT/'anova_tukey_posthoc.csv', index=False)
chi_result.to_csv(OUT/'chi_square_results.csv', index=False)
contingency.to_csv(OUT/'chi_square_contingency.csv')
monthly.to_csv(OUT/'monthly_business_metrics.csv')
forecast_test.to_csv(OUT/'forecast_test_set.csv', index=False)
performance.to_csv(OUT/'forecast_model_performance.csv', index=False)
future_forecast.to_csv(OUT/'future_revenue_forecast.csv', index=False)
print('Exports completed.')